<a href="https://colab.research.google.com/github/Phreely/AlphaFold3_pLDDT/blob/main/Boltz_2_yaml_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import requests
import yaml
from string import ascii_uppercase

# User inputs
query_sequence = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASKK'  #@param {type:"string"}
ligand_input = 'N[C@@H](Cc1ccc(O)cc1)C(=O)O'  #@param {type:"string"}
ligand_input_ccd = 'SAH'  #@param {type:"string"}
ligand_input_common_name = ''  #@param {type:"string"}
dna_input = ''  #@param {type:"string"}
jobname = 'test'  #@param {type:"string"}

#@markdown ---
#@markdown ### **Advanced Settings**
#@markdown Enter a number for `max_msa`. Leave at 0 to use Boltz-2 defaults.
max_msa = 0 #@param {type:"integer"}
num_seeds = "1" #@param ["auto", "1", "2", "4", "8", "16"]
recycling_steps = "auto" #@param ["auto", "1", "3", "5", "10"]

# 1. Clean up and Uppercase
query_sequence = re.sub(r'\s+', '', query_sequence).upper()
dna_input = re.sub(r'\s+', '', dna_input).upper()
ligand_input = re.sub(r'\s+', '', ligand_input) # SMILES are case-sensitive, do not uppercase
ligand_input_ccd = re.sub(r'\s+', '', ligand_input_ccd).upper()
ligand_input_common_name = re.sub(r'\s+', '', ligand_input_common_name)

# 2. Setup Jobname and Directory
basejobname = re.sub(r'\W+', '', jobname)
jobname = basejobname + "_" + hashlib.sha1(query_sequence.encode()).hexdigest()[:5]
os.makedirs(jobname, exist_ok=True)

# 3. Handle Common Names via PubChem
def get_smiles(compound_name):
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"
        r = requests.get(url, timeout=5)
        return r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except:
        return None

protein_sequences = query_sequence.split(':') if query_sequence else []
dna_sequences = dna_input.split(':') if dna_input else []
smiles_ligands = ligand_input.split(':') if ligand_input else []
ccd_ligands = ligand_input_ccd.split(':') if ligand_input_ccd else []

if ligand_input_common_name:
    for name in ligand_input_common_name.split(':'):
        smi = get_smiles(name)
        if smi:
            print(f"Found SMILES for {name}: {smi}")
            smiles_ligands.append(smi)

# 4. Construct YAML Dictionary
boltz_dict = {
    "name": jobname,
    "sequences": []
}

chain_gen = iter(ascii_uppercase)

# Add Proteins
for seq in protein_sequences:
    if seq:
        boltz_dict["sequences"].append({"protein": {"id": next(chain_gen), "sequence": seq}})

# Add DNA
for seq in dna_sequences:
    if seq:
        boltz_dict["sequences"].append({"dna": {"id": next(chain_gen), "sequence": seq}})

# Add SMILES Ligands into the sequences block
for i, smi in enumerate(smiles_ligands):
    if smi:
        boltz_dict["sequences"].append({"ligand": {"id": f"L{next(chain_gen)}", "smiles": smi}})

# Add CCD Ligands into the sequences block
for ccd in ccd_ligands:
    if ccd:
        boltz_dict["sequences"].append({"ligand": {"id": f"C{next(chain_gen)}", "ccd": ccd}})

# 5. Advanced Parameters
sampling = {}
if num_seeds != "auto":
    sampling["num_samples"] = int(num_seeds)
if recycling_steps != "auto":
    sampling["recycling_steps"] = int(recycling_steps)
if sampling:
    boltz_dict["sampling"] = sampling

if max_msa > 0:
    boltz_dict["msa"] = {"max_msa_seqs": max_msa}

# 6. Save and Download
yaml_path = os.path.join(jobname, f"{jobname}.yaml")
with open(yaml_path, 'w') as f:
    yaml.dump(boltz_dict, f, default_flow_style=False, sort_keys=False)

print(f"\nSuccess! YAML created at {yaml_path}")
files.download(yaml_path)

Boltz-2 YAML generated at: testhoxB_ca5a7/testhoxB_ca5a7.yaml


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>